#### Vérification et téléchargement du fichier si besoin

In [2]:
import pandas as pd

from pathlib import Path
from nutriscope import config
from nutriscope.cli import main
from nutriscope.filtering import output_path_for

DATASET_NAME = 'csv'
DATA_FILE = output_path_for(config.DEFAULT_COUNTRY, DATASET_NAME)

if DATA_FILE.exists():
    print(f'{DATA_FILE.name} présent, rien à faire')
else:
    print(f'{DATA_FILE.name} absent : téléchargement et extraction (plusieurs minutes)...')  
    main(['prepare', '--only', DATASET_NAME])  

if not DATA_FILE.exists():
    raise FileNotFoundError(f'{DATA_FILE} toujours absent après `prepare`.')

food_france.csv.gz absent : téléchargement et extraction (plusieurs minutes)...
Dernière modification serveur : Fri, 30 Oct 2026 09:08:40 GMT


food.csv.gz: 100%|██████████| 1.42G/1.42G [04:11<00:00, 6.06MB/s]


OK [csv] : C:\Users\Alex\Documents\www\projet_nutriscope\data\food.csv.gz (1.53 Go)


Lecture CSV: 4.88Mligne [12:52, 6.32kligne/s, kept=1,292,998]

OK [csv] : 1,292,998 produits (en:france) dans C:\Users\Alex\Documents\www\projet_nutriscope\data\france\food_france.csv.gz (459 Mo)


#### Import du fichier CSV et des colonnes

In [3]:
CSV_GZ = Path("..") / "data" / "france" / "food_france.csv.gz"

columns = config.COLUMNS

dataframe = pd.read_csv(
    CSV_GZ,
    sep="\t",
    usecols=columns,
    dtype={"code": "string"},
    low_memory=False
)

##### Distribution des valeurs sur les nutriments
Ici, on récupère : 
* Les quartiles
* Les valeurs min et max
* La moyenne
* La médiane

Ca nous permet d'identifier facilement les valeurs aberrantes

In [6]:
num = dataframe.select_dtypes(include="number")

table = pd.DataFrame({
    "0.25": num.quantile(0.25),
    "0.50": num.quantile(0.50),
    "0.75": num.quantile(0.75),
    "min": num.min(),
    "max": num.max(),
    "moyenne": num.mean(),
    "médiane": num.median(),
})
table.index.name = "nutriment"

table.style.format("{:.1f}")

,0.25,0.50,0.75,min,max,moyenne,médiane
nutriment,,,,,,,
nutriscore_score,3.0,11.0,19.0,-17.0,53.0,11.2,11.0
completeness,0.3,0.4,0.6,0.1,1.1,0.4,0.4
energy-kj_100g,483.8,1094.7,1674.5,-700.0,10000000000.0,12179.6,1094.7
energy-kcal_100g,113.0,261.0,400.0,-113.9,14232696253056200.0,15531723246.7,261.0
energy_100g,476.5,1088.0,1671.7,-700.0,10000000000.0,12075.0,1088.0
fat_100g,1.0,8.1,22.2,-0.0,856535223453253001384139073764652281344540278784.0,941892332281609348816668473248765081288704.0,8.1
saturated-fat_100g,0.2,2.0,8.0,-3.6,1238.0,5.4,2.0
carbohydrates_100g,2.7,14.0,51.0,0.0,40000.0,27.0,14.0
sugars_100g,0.6,3.5,19.3,0.0,1174717414144440050270536724381696.0,1292358196010922230543286272.0,3.5


##### Distribution et cardinalité sur les nutriscores
On récupère la part des produits notés, et des non notés

In [7]:
grades = dataframe['nutriscore_grade']

scored = grades.isin(list(config.NUTRISCORE_GRADES))

summary = pd.Series({
    "noté (a à e)": scored.sum(),
    "unknown": (grades == "unknown").sum(),
    "not-applicable": (grades == "not-applicable").sum(),
    "manquant (NaN)": grades.isna().sum(),
})

pd.DataFrame(
    {"nombre": summary, "%": summary / len(grades) * 100}).style.format(
    {"nombre": "{:,.0f}", "%": "{:.1f}"}
)

,nombre,%
noté (a à e),"473,607",36.6
unknown,"745,838",57.7
not-applicable,"43,114",3.3
manquant (NaN),"30,439",2.4


Ici, on analyse un peu plus en détail.
On fait ressortir le pourcentage de chaque note sur l'ensemble des produits qui ont une notes

In [8]:
counts_scored = grades[scored].value_counts().sort_index()

pd.DataFrame({
    "nombre": counts_scored,
    "% des notés": counts_scored / counts_scored.sum() * 100,
}).style.format({"nombre": "{:,.0f}", "% des notés": "{:.1f}"})

,nombre,% des notés
nutriscore_grade,,
a,"64,610",13.6
b,"50,288",10.6
c,"98,825",20.9
d,"125,110",26.4
e,"134,774",28.5


##### Récupération des doublons de codes-barres

In [9]:
codes = dataframe["code"]
is_dup = codes.duplicated(keep=False) & codes.notna()

print(f"Codes manquants          : {codes.isna().sum():,}")
print(f"Lignes avec code répété  : {is_dup.sum():,}")
print(f"Codes distincts répétés  : {codes[is_dup].nunique():,}")

Codes manquants          : 0
Lignes avec code répété  : 46
Codes distincts répétés  : 23


In [10]:
exact = dataframe.duplicated(keep=False)
print(f"Lignes identiques sur toutes les colonnes : {exact.sum():,}")

Lignes identiques sur toutes les colonnes : 0


#### Décision sur le traitement des boublons :
* Supprimer les doublons pour les lignes qui sont complètement identiques
* Pour celles qui ont un code en doublons mais des infos différentes, garder la plus complète

In [11]:
dataframe[is_dup].sort_values("code").head(10)

,code,product_name,brands,categories,categories_tags,labels,countries,countries_tags,allergens,nutriscore_score,...,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits-vegetables-legumes_100g
4219,0059527070430,Goji Snack Bars,Taste Of Nature,"Snacks, Snacks sucrés, Barres, Barres de céréales","en:snacks,en:sweet-snacks,en:bars,en:cereal-bars","No gluten, Organic, Vegetarian, Kosher, Vegan,...","Canada, France, World","en:canada,en:france,en:world","en:nuts,en:peanuts",NaN,...,1730.0,17.3,NaN,48.0,NaN,NaN,16.7,NaN,NaN,NaN
1062737,0059527070430,Goji,Taste of Nature,"Snacks,Snacks sucrés,Barres,Barres aux fruits","en:snacks,en:sweet-snacks,en:bars,fr:barres-au...","Sans gluten,Bio,Végétarien,Ecocert,Bio europée...",France,en:france,"en:nuts,en:peanuts",6.0,...,1922.0,28.0,3.6,38.0,27.0,7.6,13.0,0.10,0.040,NaN
4224,0059527171687,Tast Of Nature Pomegranate Granenreep @,Taste of nature,NaN,NaN,"No gluten, No GMOs, Non GMO project",France,en:france,NaN,NaN,...,1816.0,25.0,3.3,49.0,25.0,5.4,11.0,0.00,0.000,NaN
1062738,0059527171687,Pomegranate,Taste of Nature,"Snacks,Snacks sucrés,Barres,Barres aux fruits","en:snacks,en:sweet-snacks,en:bars,fr:barres-au...","Sans gluten,Bio,Végétarien,Ecocert,Bio europée...",France,en:france,"en:nuts,en:peanuts",7.0,...,1845.0,25.0,3.3,43.0,27.0,6.1,10.0,0.04,0.016,NaN
4237,0059527401555,Almond Fruit And Nut Snack Bar,Taste Of Nature,"Snacks, Snacks sucrés, Barres, Snacks; barres ...","en:snacks,en:sweet-snacks,en:bars,fr:snacks-ba...","Sans gluten, Bio, en:Certified gluten-free, No...","Belgique,France,Suisse, World","en:france,en:world,en:belgique,en:suisse",NaN,7.0,...,1989.0,25.0,2.7,40.0,26.0,8.2,11.8,0.04,0.016,NaN
1062732,0059527401555,Almond,Taste of Nature,"Snacks,Snacks sucrés,Barres,Barres aux fruits","en:snacks,en:sweet-snacks,en:bars,fr:barres-au...","Sans gluten,Bio,Végétarien,Ecocert,Bio europée...",France,en:france,"en:nuts,en:peanuts",6.0,...,1870.0,25.0,2.7,40.0,26.0,8.2,12.0,0.04,0.016,NaN
4238,0059527501552,Cranberry Fruit and Nut Snack Bar,Taste Of Nature,"Aliments et boissons à base de végétaux, Alime...","en:plant-based-foods-and-beverages,en:plant-ba...","No gluten, Organic, Vegetarian, Vegan, Vegan A...","France,Suisse, World","en:france,en:world,en:suisse","en:nuts,en:sesame-seeds",7.0,...,1933.0,27.0,3.6,38.0,24.0,78.0,13.6,0.03,0.012,NaN
1062728,0059527501552,Cranberry,Taste of Nature,"Snacks,Snacks sucrés,Barres,Barres aux fruits","en:snacks,en:sweet-snacks,en:bars,fr:barres-au...","Sans gluten,Bio,Végétarien,Ecocert,Bio europée...",France,en:france,"en:nuts,en:peanuts",7.0,...,1935.0,27.0,3.6,38.0,24.0,7.8,14.0,0.03,0.012,NaN
4240,0059527601559,"Coconut, Cranberry, Brazil Nut Snack Bar Varie...",Taste Of Nature,"Aliments et boissons à base de végétaux, Alime...","en:plant-based-foods-and-beverages,en:plant-ba...","No gluten, Organic, Vegetarian, Vegan, Vegan A...","France,Switzerland, World","en:france,en:switzerland,en:world","en:nuts,en:sesame-seeds,en:soybeans",15.0,...,2015.0,27.0,4.2,49.0,28.0,5.7,14.0,0.00,0.000,NaN
1062735,0059527601559,Brazil nut,Taste of Nature,"Snacks,Snacks sucrés,Barres,Barres aux fruits","en:snacks,en:sweet-snacks,en:bars,fr:barres-au...","Sans gluten,Bio,Végétarien,Ecocert,Bio europée...",France,en:france,"en:nuts,en:peanuts",7.0,...,1925.0,28.0,4.3,40.0,26.0,6.0,12.0,0.03,0.012,NaN


##### Taux de remplissage sur les énergies

In [12]:
kj = dataframe["energy-kj_100g"]
kcal = dataframe["energy-kcal_100g"]

both_filled = kj.notna() & kcal.notna()

print(f"Les deux remplis : {both_filled.sum():,} ({both_filled.mean():.1%} des produits)")
print(f"Seulement kJ    : {(kj.notna() & kcal.isna()).sum():,}")
print(f"Seulement kcal  : {(kj.isna() & kcal.notna()).sum():,}")
print(f"Aucun des deux  : {(kj.isna() & kcal.isna()).sum():,}")

Les deux remplis : 908,087 (70.2% des produits)
Seulement kJ    : 378
Seulement kcal  : 8,276
Aucun des deux  : 376,257


##### Contrôles des incohérences d'unités sur les énergies
Ratio kJ/kcal théorique : 4,184. Plage retenue : 3,8 à 4,6.
Choisie car une plage 4,0-4,4 laissait de côté ~61 000 lignes proches (arrondis d'étiquettes).
Les lignes avec une énergie à 0 sont exclues ici (traitées dans « valeurs impossibles »).

In [13]:
both = dataframe[["energy-kj_100g", "energy-kcal_100g"]].dropna()
both = both[(both > 0).all(axis=1)]

ratio = both["energy-kj_100g"] / both["energy-kcal_100g"]
ok = ratio.between(*config.KJ_KCAL_RATIO_RANGE)

print(f"{len(both):,} lignes comparables")
print(f"  cohérentes   : {ok.sum():,} ({ok.mean():.1%})")
print(f"  incohérentes : {(~ok).sum():,} ({(~ok).mean():.1%})")

889,869 lignes comparables
  cohérentes   : 798,813 (89.8%)
  incohérentes : 91,056 (10.2%)


##### Contrôle des valeurs impossibles
1. Les énergies - kJ / kcal

In [14]:
kj = dataframe["energy-kj_100g"]
kcal = dataframe["energy-kcal_100g"]

energy_rules = {
    "kJ négatif": (kj < 0).sum(),
    "kcal négatif": (kcal < 0).sum(),
    "kJ = 0": (kj == 0).sum(),
    "kcal = 0": (kcal == 0).sum(),
    f"kJ > {config.KJ_MAX}": (kj > config.KJ_MAX).sum(),
    f"kcal > {config.KCAL_MAX}": (kcal > config.KCAL_MAX).sum(),
}

energy_table = pd.Series(energy_rules, name="lignes").to_frame()
energy_table["% de toutes les lignes"] = energy_table["lignes"] / len(dataframe) * 100
energy_table.style.format({"lignes": "{:,.0f}", "% de toutes les lignes": "{:.2f}"})

,lignes,% de toutes les lignes
kJ négatif,8,0.00
kcal négatif,1,0.00
kJ = 0,"14,908",1.15
kcal = 0,"18,151",1.40
kJ > 3700,680,0.05
kcal > 900,"1,007",0.08


In [15]:
filled = dataframe[["energy-kj_100g", "energy-kcal_100g"]].dropna()
zeros = (filled == 0).sum(axis=1)

print(f"Les deux à 0 (plausible pour de l'eau) : {(zeros == 2).sum():,}")
print(f"Un seul à 0 (impossible)               : {(zeros == 1).sum():,}")

Les deux à 0 (plausible pour de l'eau) : 11,510
Un seul à 0 (impossible)               : 6,700


2. Les nutriments 

In [16]:
# Nutriments exprimés en g/100 g (tous sauf les énergies, qui sont en kJ ou kcal)
mass_nutrients = [n for n in config.NUTRIENTS if not n.startswith("energy")]
over_max_label = f"> {config.MAX_PER_100G}"

mass_sub = dataframe[mass_nutrients]

mass_table = pd.DataFrame({
    "renseignées": mass_sub.notna().sum(),
    "négatives": (mass_sub < 0).sum(),
    over_max_label: (mass_sub > config.MAX_PER_100G).sum(),
})
mass_table["% négatives"] = mass_table["négatives"] / mass_table["renseignées"] * 100
mass_table[f"% {over_max_label}"] = mass_table[over_max_label] / mass_table["renseignées"] * 100
mass_table.index.name = "nutriment"

mass_table.style.format({
    "renseignées": "{:,.0f}", "négatives": "{:,.0f}", over_max_label: "{:,.0f}",
    "% négatives": "{:.2f}", f"% {over_max_label}": "{:.2f}",
})

,renseignées,négatives,> 100,% négatives,% > 100
nutriment,,,,,
fat_100g,"909,377",1,47,0.00,0.01
saturated-fat_100g,"907,730",1,16,0.00,0.00
sugars_100g,"908,972",0,55,0.00,0.01
fiber_100g,"398,674",19,22,0.00,0.01
proteins_100g,"911,018",1,36,0.00,0.00
salt_100g,"852,022",0,55,0.00,0.01
carbohydrates_100g,"909,660",0,95,0.00,0.01
sodium_100g,"852,023",0,41,0.00,0.00
fruits-vegetables-legumes_100g,"358,221",52,"2,147",0.01,0.60


In [17]:
sugars_vs_carbs = dataframe[["sugars_100g", "carbohydrates_100g"]].dropna()
saturated_vs_fat = dataframe[["saturated-fat_100g", "fat_100g"]].dropna()
macros = dataframe[["fat_100g", "carbohydrates_100g", "proteins_100g", "salt_100g"]].dropna()

tol = config.LABEL_ROUNDING_TOL

consistency_rules = {
    "sucres > glucides": (
        (sugars_vs_carbs["sugars_100g"] > sugars_vs_carbs["carbohydrates_100g"] + tol).sum(),
        len(sugars_vs_carbs),
    ),
    "saturés > graisses": (
        (saturated_vs_fat["saturated-fat_100g"] > saturated_vs_fat["fat_100g"] + tol).sum(),
        len(saturated_vs_fat),
    ),
    f"graisses+glucides+protéines+sel > {config.MACRO_SUM_MAX}": (
        (macros.sum(axis=1) > config.MACRO_SUM_MAX).sum(),
        len(macros),
    ),
}

consistency_table = pd.DataFrame(consistency_rules, index=["violations", "comparables"]).T
consistency_table["%"] = consistency_table["violations"] / consistency_table["comparables"] * 100
consistency_table.style.format({"violations": "{:,.0f}", "comparables": "{:,.0f}", "%": "{:.2f}"})

,violations,comparables,%
sucres > glucides,452,"902,796",0.05
saturés > graisses,255,"900,366",0.03
graisses+glucides+protéines+sel > 105,"1,245","840,035",0.15


#### Contrôle sur le sel
Très peu de produits ont des valeurs incohérentes sur le sel / sodium.
On peut les exclure du dataset sans se priver de beaucoup de données

In [18]:
salt_sodium = dataframe[["salt_100g", "sodium_100g"]].dropna()

expected_salt = salt_sodium["sodium_100g"] * config.SALT_PER_SODIUM
salt_gap = (salt_sodium["salt_100g"] - expected_salt).abs()
salt_tolerance = (expected_salt * config.SALT_SODIUM_REL_TOL).clip(lower=config.SALT_SODIUM_ABS_TOL)
salt_inconsistent = salt_gap > salt_tolerance

print(f"{len(salt_sodium):,} lignes comparables")
print(f"  cohérentes   : {(~salt_inconsistent).sum():,} ({(~salt_inconsistent).mean():.1%})")
print(f"  incohérentes : {salt_inconsistent.sum():,} ({salt_inconsistent.mean():.1%})")

852,022 lignes comparables
  cohérentes   : 848,467 (99.6%)
  incohérentes : 3,555 (0.4%)


#### Cardinalité

In [19]:
text_columns = dataframe.select_dtypes(exclude="number")

cardinality_table = pd.DataFrame({
    "distinctes": text_columns.nunique(),
    "% manquants": text_columns.isna().mean() * 100,
})
cardinality_table.index.name = "colonne"

cardinality_table.sort_values("distinctes").style.format(
    {"distinctes": "{:,.0f}", "% manquants": "{:.1f}"}
)

,distinctes,% manquants
colonne,,
nutriscore_grade,7,2.4
pnns_groups_1,12,2.3
pnns_groups_2,43,2.3
food_groups,48,56.4
food_groups_tags,48,56.4
allergens,236,99.9
countries,"1,287",40.6
countries_tags,"4,288",0.0
labels,"22,406",82.5


#### Répartition des notes par rayon

In [20]:
grades_in_order = list(config.NUTRISCORE_GRADES)
scored_products = dataframe.loc[scored, ["pnns_groups_1", "nutriscore_grade"]]

grade_counts = (
    pd.crosstab(scored_products["pnns_groups_1"], scored_products["nutriscore_grade"])
    .reindex(columns=grades_in_order, fill_value=0)
)
total_scored = grade_counts.sum(axis=1)

grade_by_group = (
    grade_counts.div(total_scored, axis=0).mul(100)
    .assign(**{"total notés": total_scored})
    .sort_values("total notés", ascending=False)
)
grade_by_group.index.name = "rayon (pnns_groups_1)"

(
    grade_by_group.style
    .format({**{grade: "{:.1f} %" for grade in grades_in_order}, "total notés": "{:,.0f}"})
    .background_gradient(cmap="Blues", subset=grades_in_order, axis=1)
)

nutriscore_grade,a,b,c,d,e,total notés
rayon (pnns_groups_1),,,,,,
Sugary snacks,0.8 %,1.2 %,5.2 %,26.3 %,66.5 %,"94,774"
Fish Meat Eggs,21.0 %,9.1 %,12.9 %,28.7 %,28.4 %,"78,894"
Milk and dairy products,4.7 %,7.8 %,30.8 %,46.8 %,9.9 %,"54,906"
Cereals and potatoes,31.5 %,18.9 %,27.2 %,17.7 %,4.7 %,"48,179"
unknown,15.4 %,10.3 %,21.8 %,25.6 %,26.9 %,"38,370"
Composite foods,9.6 %,15.8 %,47.1 %,23.3 %,4.2 %,"36,042"
Beverages,6.1 %,20.6 %,29.4 %,17.2 %,26.7 %,"33,367"
Fat and sauces,4.5 %,20.3 %,21.1 %,20.1 %,33.9 %,"31,681"
Fruits and vegetables,48.4 %,11.9 %,20.0 %,13.3 %,6.4 %,"29,990"


#### Nombre de produits par rayons

In [21]:
by_group = (
    dataframe
    .assign(is_scored=scored)
    .groupby("pnns_groups_1", dropna=False)
    .agg(produits=("code", "size"), part_notes=("is_scored", "mean"))
    .sort_values("produits", ascending=False)
)

by_group.style.format({"produits": "{:,.0f}", "part_notes": "{:.1%}"})

,produits,part_notes
pnns_groups_1,,
unknown,"700,904",5.5%
Sugary snacks,"119,562",79.3%
Fish Meat Eggs,"98,871",79.8%
Milk and dairy products,"64,649",84.9%
Cereals and potatoes,"58,299",82.6%
Beverages,"47,443",70.3%
Fruits and vegetables,"41,078",73.0%
Fat and sauces,"40,124",79.0%
Composite foods,"40,106",89.9%


In [22]:
completeness = dataframe[config.COMPLETENESS_NUTRIENTS].notna().sum(axis=1)
completeness_counts = completeness.value_counts().sort_index()

completeness_table = pd.DataFrame({
    "produits": completeness_counts,
    "%": completeness_counts / completeness_counts.sum() * 100,
})
completeness_table["produits avec au moins N"] = completeness_table["produits"][::-1].cumsum()[::-1]
completeness_table["% avec au moins N"] = (
    completeness_table["produits avec au moins N"] / len(completeness) * 100
)
completeness_table.index.name = f"nutriments renseignés (sur {len(config.COMPLETENESS_NUTRIENTS)})"

completeness_table.style.format({
    "produits": "{:,.0f}",
    "%": "{:.1f}",
    "produits avec au moins N": "{:,.0f}",
    "% avec au moins N": "{:.1f}",
})

,produits,%,produits avec au moins N,% avec au moins N
nutriments renseignés (sur 7),,,,
0,"363,114",28.1,"1,292,998",100.0
1,"12,180",0.9,"929,884",71.9
2,"5,639",0.4,"917,704",71.0
3,"3,165",0.2,"912,065",70.5
4,"3,213",0.2,"908,900",70.3
5,"43,363",3.4,"905,687",70.0
6,"494,732",38.3,"862,324",66.7
7,"367,592",28.4,"367,592",28.4


#### Complétion sur le score et la note nutriscore

In [23]:
has_grade = dataframe["nutriscore_grade"].isin(list(config.NUTRISCORE_GRADES))
has_score = dataframe["nutriscore_score"].notna()

grade_label = has_grade.map({True: "Note a–e", False: "Pas de note"}).rename("")
score_label = has_score.map({True: "Score renseigné", False: "Pas de score"}).rename("")

ROWS = ["Note a–e", "Pas de note", "Total"]
COLS = ["Score renseigné", "Pas de score", "Total"]

grade_vs_score_counts = pd.crosstab(grade_label, score_label, margins=True, margins_name="Total").loc[ROWS, COLS]
grade_vs_score_share = pd.crosstab(
    grade_label, score_label, margins=True, margins_name="Total", normalize="all"
).loc[ROWS, COLS] * 100

# Une cellule = "nombre (part du total)"
grade_vs_score = grade_vs_score_counts.map("{:,.0f}".format) + " (" + grade_vs_score_share.map("{:.1f} %".format) + ")"
grade_vs_score

,Score renseigné,Pas de score,Total
,,,
Note a–e,"473,607 (36.6 %)",0 (0.0 %),"473,607 (36.6 %)"
Pas de note,0 (0.0 %),"819,391 (63.4 %)","819,391 (63.4 %)"
Total,"473,607 (36.6 %)","819,391 (63.4 %)","1,292,998 (100.0 %)"
